In [1]:
import math
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from tqdm import tqdm

In [4]:
class SinusoidalTimeEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, t):
        # t: (B,) in [0,1]
        half = self.dim // 2
        freqs = torch.exp(-math.log(10000) * torch.arange(0, half, dtype=torch.float32) / half).to(t.device)
        args = t[:, None] * freqs[None, :] * 2 * math.pi
        emb = torch.cat([torch.sin(args), torch.cos(args)], dim=-1)
        if self.dim % 2 == 1:
            emb = torch.cat([emb, torch.zeros(t.shape[0], 1, device=t.device)], dim=-1)
        return emb  # (B, dim)
    
time_embedding = SinusoidalTimeEmbedding(32)

t = torch.rand(32, 1)
time_embedding(t).shape

torch.Size([32, 1, 32])

In [ ]:
class SmallUNet(nn.Module):
    def __init__(self, in_channels=3, base_ch=64, time_emb_dim=128):
        super().__init__()
        self.time_mlp = nn.Sequential(
            SinusoidalTimeEmbedding(time_emb_dim),
            nn.Linear(time_emb_dim, time_emb_dim),
            nn.SiLU(),
            nn.Linear(time_emb_dim, base_ch)
        )
        # encoder
        self.conv1 = nn.Conv2d(in_channels, base_ch, 3, padding=1)
        self.conv2 = nn.Conv2d(base_ch, base_ch*2, 3, padding=1, stride=2)
        self.conv3 = nn.Conv2d(base_ch*2, base_ch*4, 3, padding=1, stride=2)
        # decoder
        self.deconv1 = nn.ConvTranspose2d(base_ch*4, base_ch*2, 4, stride=2, padding=1)
        self.deconv2 = nn.ConvTranspose2d(base_ch*2, base_ch, 4, stride=2, padding=1)
        self.out = nn.Conv2d(base_ch, in_channels, 3, padding=1)

        self.act = nn.SiLU()
    